**Table of contents**<a id='toc0_'></a>    
- [Install Libraries](#toc1_)    
- [Import Libraries](#toc2_)    
- [Connect DB](#toc3_)    
    - [ Google Colab](#toc3_1_1_)    
    - [Local](#toc3_1_2_)    
- [Import Data](#toc4_)    
- [Get Data](#toc5_)    
- [Strategy](#toc6_)    
- [Backtest](#toc7_)    
- [Visualization](#toc8_)    
- [Execution](#toc9_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Install Libraries](#toc0_)

In [ ]:
!pip install -q ezyquant

# <a id='toc2_'></a>[Import Libraries](#toc0_)

In [ ]:
import numpy as np
import pandas as pd

import ezyquant as ez

from ezyquant.backtesting import Context
from ezyquant import fields as fld
from ezyquant.reader import _SETDataReaderCached

# <a id='toc3_'></a>[Connect DB](#toc0_)

### <a id='toc3_1_1_'></a>[ Google Colab](#toc0_)

In [ ]:
# google authen
from google.colab import drive
drive.mount('/content/drive')

# connect DB
database_path = "/content/drive/MyDrive/DB/ezyquant.db" # GoogleColab
ez.connect_sqlite(database_path)

In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

### <a id='toc3_1_2_'></a>[Local](#toc0_)

In [ ]:
# Connect Local DB
# database_path = "/path/to/ezyquant.db" # ระบุ path ของ ezyquant.db
# ez.connect_sqlite(database_path)

# <a id='toc4_'></a>[Import Data](#toc0_)

Stock universe data

In [ ]:
sdr = _SETDataReaderCached()
start_data = "2008-01-01"  # start date import data
start_date = "2010-01-01"  # start backtest date
end_date = sdr.last_update()

ssc = ez.SETSignalCreator(
    start_date=start_data,
    end_date=end_date,
    index_list=["SET100"],
)

Benchmark

In [ ]:
set_index_df = sdr.get_data_index_daily(
                field=fld.D_INDEX_CLOSE,
                index_list=[fld.MARKET_SET],
                start_date=start_data,
                end_date=end_date,
            )
set_index_return = set_index_df.pct_change().fillna(0.0)

Backtest Properties

In [ ]:
PCT_COMMISSION = 0.25
PCT_BUY_SLIP = 0.01
PCT_SELL_SLIP = 0.01

# <a id='toc5_'></a>[Get Data](#toc0_)

Website: https://www.ezyquant.com/

Docs: https://pydoc.ezyquant.com/user_guide/parameter_property.html

In [ ]:
# ohlc
close_df = ssc.get_data(field="close", timeframe="daily")
value_df = ssc.get_data(field="value", timeframe="daily")

pe_df = ssc.get_data(field="pe", timeframe='daily')

# <a id='toc6_'></a>[Strategy](#toc0_)

In [ ]:
sma_50_df = ssc.ta.sma(close_df, 50)
sma_200_df = ssc.ta.sma(close_df, 200)

above_10d_cond = close_df > close_df.shift(10)
sma_cond = close_df > sma_200_df
value_cond = value_df.rolling(window=20).mean() > 2e7  # volume > 2e7 (20m)
pe_cond = (pe_df > 10) & (pe_df < 20)


# index
set_sma_df = ssc.ta.sma(set_index_df, 50)
set_index_cond = set_index_df > set_sma_df

Combine Signal

In [ ]:
signal_cond = above_10d_cond & sma_cond & value_cond & pe_cond

# filter index signal
signal_cond = pd.DataFrame(
    np.where((set_index_cond), signal_cond, -100),
    columns=signal_cond.columns,
    index=signal_cond.index,
)

In [ ]:
signal_cond

Screen universe

In [ ]:
signal_df = ssc.screen_universe(signal_cond, mask_value=-99)

In [ ]:
signal_df

Ranking

In [ ]:
# -- filter price up trend by sma --
price_above_ma = pd.DataFrame(
    np.where((signal_df == True), (close_df / close_df.shift(10)), np.nan),
    columns=signal_df.columns,
    index=signal_df.index,
)

pos_num = 10
signal_trade = ssc.rank(factor_df=price_above_ma, quantity=pos_num, ascending=False)

In [ ]:
signal_trade

Handle sign signal

trading sings: https://www.set.or.th/th/market/information/trading-procedure/trading-signs

In [ ]:
lookahead_signal = ssc.screen_universe(close_df, mask_value=-500).shift(-5)

final_signal = pd.DataFrame(
    np.where(lookahead_signal == -1, -1, signal_trade),
    columns=signal_trade.columns,
    index=signal_trade.index,
)

# <a id='toc7_'></a>[Backtest](#toc0_)

In [ ]:
# -- Backtest Algorithm --
def backtest_algorithm(c: Context):
    if c.volume == 0 and c.signal > 0:
        return c.target_pct_port(1 / pos_num)
    elif c.volume > 0 and c.signal < 0:
        return c.target_pct_port(0)  # sell
    return 0


# -- Backtest Setting --
result = ez.backtest(
    signal_df=final_signal,
    backtest_algorithm=backtest_algorithm,
    start_date=start_date,
    end_date=end_date,
    initial_cash=1e8, # 10m
    pct_commission=PCT_COMMISSION,
    pct_buy_slip=PCT_BUY_SLIP,
    pct_sell_slip=PCT_SELL_SLIP,
    price_match_mode="weighted",
    signal_delay_bar=1,
)

# <a id='toc8_'></a>[Visualization](#toc0_)

In [ ]:
result.to_basic(benchmark=set_index_return)

In [ ]:
result.cagr

# <a id='toc9_'></a>[Execution](#toc0_)

In [ ]:
final_signal.iloc[-1].sort_values().head(10)

In [ ]:
# Get symbol list to execution
# 1. Get the latest date
# 2. Drop na
# 3. transform to list
symbol_signal_list = final_signal.iloc[-1].sort_values() \
                    .dropna() \
                    .index.to_list()
symbol_signal_list

Install Libraries

In [ ]:
!pip install -q ezyquant-execution

Import Libraries

In [ ]:
from ezyquant_execution.context import (
    Investor,
    ExecuteContext,
    ExecuteContextSymbol
)
from ezyquant_execution.entity import (
    SIDE_BUY,
    SIDE_SELL,
    SIDE_TYPE,
)
from ezyquant_execution.utils import match_tick_price

import os

In [ ]:
SIDE_BUY

Connect Settrade

In [ ]:
from google.colab import userdata
import os # Make sure this is imported for os.environ to work!
# Colab
SETTRADE_APP_ID = userdata.get("SETTRADE_APP_ID")
SETTRADE_APP_SECRET = userdata.get("SETTRADE_APP_SECRET")
SETTRADE_APP_CODE = userdata.get("SETTRADE_APP_CODE")
SETTRADE_BROKER_ID = userdata.get("SETTRADE_BROKER_ID")
SETTRADE_PIN = userdata.get("SETTRADE_PIN")

SETTRADE_ACCOUNT_NO = "pupuchon-E"

# Local
# SETTRADE_APP_ID = os.environ["SETTRADE_APP_ID"]
# SETTRADE_APP_SECRET = os.environ["SETTRADE_APP_SECRET"]
# SETTRADE_APP_CODE = os.environ["SETTRADE_APP_CODE"]
# SETTRADE_BROKER_ID = os.environ["SETTRADE_BROKER_ID"]
# SETTRADE_PIN = os.environ["SETTRADE_PIN"]

# SETTRADE_ACCOUNT_NO = os.environ["SETTRADE_ACCOUNT_NO"]


In [ ]:
settrade_equity_user = Investor(
    app_id=SETTRADE_APP_ID,
    app_secret=SETTRADE_APP_SECRET,
    app_code=SETTRADE_APP_CODE,
    broker_id=SETTRADE_BROKER_ID,
)

equity_ctx = ExecuteContext(settrade_user=settrade_equity_user, account_no=SETTRADE_ACCOUNT_NO, pin=SETTRADE_PIN)
equity_ctx.port_value


In [ ]:
def _handle_price(
    equity_exe_ctx_symbol: ExecuteContextSymbol, side: SIDE_TYPE
) -> float:
    """Retrieve the Best Cheapest cost based on the side, considering:

        - The Best Bid + 1 or the cheapest price on the BUY side
        - The Ask - 1 or the cheapest price on the SELL side
    This is accomplished using the Limit mode or Normal BUY/SELL.
    """
    if side == SIDE_BUY:
        best_price = equity_exe_ctx_symbol.best_bid_price
    elif side == SIDE_SELL:
        best_price = equity_exe_ctx_symbol.best_ask_price
    else:
        msg = "Invalid Trade Side"
        raise ValueError(msg)

    if side == SIDE_BUY:
        return match_tick_price(best_price, 1)
    else:
        return match_tick_price(best_price, -1)

Place buy order

In [ ]:
# NOTE: In the UAT environment, liquidity is not available, resulting in a best bid/ask price of 0.
for symbol in symbol_signal_list:
    try:
        equity_exe_ctx_symbol = ExecuteContextSymbol(
            settrade_user=settrade_equity_user,
            account_no=SETTRADE_ACCOUNT_NO,
            symbol=symbol,
            pin=SETTRADE_PIN
        )
        # get best bid/ask price
        price = _handle_price(equity_exe_ctx_symbol=equity_exe_ctx_symbol, side=SIDE_BUY)
        # ...
        equity_exe_ctx_symbol.place_order(side=SIDE_BUY,
                                          volume=100,
                                          price=price,
                                          price_type="Limit",
                                          validity_type="Day"
                                          )
        print(f'Execute {symbol} at price: {price}')
    except Exception as e:
        print(f'{symbol}: {e}')
        continue

Place Sell order

In [ ]:
portfolio_list = equity_ctx.get_portfolios().portfolio_list

for position in portfolio_list:
    try:
        equity_context_symbol = ExecuteContextSymbol(settrade_user=settrade_equity_user, account_no=SETTRADE_ACCOUNT_NO, symbol=position.symbol)
        price = _handle_price(equity_exe_ctx_symbol=equity_context_symbol, side=SIDE_SELL) # best bid ask price

        equity_context_symbol.place_order(
            side=SIDE_SELL,  # type: ignore
            price_type="Limit",
            price=price,
            volume=position.current_volume, # `current_volume` is represent a volume does not pending.
            )
        print(f'Execute {symbol} at price: {price}')
    except Exception as e:
        print(f'{symbol}: {e}')
        continue

Cancel all order

In [ ]:
for symbol in symbol_signal_list:
    try:
        equity_exe_ctx_symbol = ExecuteContextSymbol(
                settrade_user=settrade_equity_user,
                account_no=SETTRADE_ACCOUNT_NO,
                symbol=symbol,
                pin=SETTRADE_PIN
            )
        equity_exe_ctx_symbol.cancel_orders()
    except Exception as e:
        print(f'{symbol}: {e}')
        continue